# 4D-MAP Phase 2 Log Analysis

Compare three runs:
- **baseline** — static `--scenario default`, no dynamic tc
- **delay** — `--dynamic-delay-profile`, path-B delay steps
- **loss** — `--dynamic-loss-profile`, path-B loss steps

Key `[utility]` fields:  
`G` throughput component · `D` delay component · `L` loss component · `U` total utility  
`bw_mbps` · `owd_ms` · `loss` · `gain` · `backoff` · `trend_ms`

In [ ]:
import sys, os
from pathlib import Path

# make parse_logs importable
REPO = Path("../..").resolve()  # repo root when notebook is at scripts/analyze/
sys.path.insert(0, str(Path(".").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from parse_logs import load_pull_log, load_tc_log, find_run_dirs, load_phase2_triple

## 1  Configure paths

Edit the cell below to point to your three `vm_run_*` directories.  
If you only have one run so far, set **SINGLE_RUN** mode.

In [ ]:
# --- EDIT HERE ---
LOGS_ROOT = REPO / "logs_exp"

# For single-run inspection (e.g. the existing 20260402_161105 experiment):
SINGLE_RUN_DIR = LOGS_ROOT / "vm_run_20260402_161105"

# For Phase 2 triple comparison (fill once you have run the matrix script):
BASELINE_DIR = LOGS_ROOT / "vm_run_BASELINE"   # replace with actual RUN_ID
DELAY_DIR    = LOGS_ROOT / "vm_run_DELAY"
LOSS_DIR     = LOGS_ROOT / "vm_run_LOSS"

SMOOTHING_SEC = 3  # rolling average window (seconds)
ACTIVE_PATHS = [0, 1, 3]  # which path IDs to show

# detect single vs triple mode automatically
TRIPLE_MODE = BASELINE_DIR.exists() and DELAY_DIR.exists() and LOSS_DIR.exists()
print(f"TRIPLE_MODE={TRIPLE_MODE}  SINGLE={SINGLE_RUN_DIR.name}")

## 2  Parse logs

In [ ]:
def smooth(s, w=SMOOTHING_SEC):
    return s.rolling(window=w, min_periods=1, center=True).mean()

if TRIPLE_MODE:
    df_util, df_mon, tc_steps = load_phase2_triple(BASELINE_DIR, DELAY_DIR, LOSS_DIR)
    labels = ["baseline", "delay", "loss"]
    print(f"utility rows: {len(df_util)}   monitor rows: {len(df_mon)}")
    print(f"tc steps: {list(tc_steps.keys())}")
else:
    pull_log = next(SINGLE_RUN_DIR.glob("pull_*.log"))
    tc_delay = next(SINGLE_RUN_DIR.glob("tc_delay_*.log"), None)
    tc_loss  = next(SINGLE_RUN_DIR.glob("tc_loss_*.log"),  None)
    label = "delay" if tc_delay else ("loss" if tc_loss else "baseline")
    df_util, df_mon = load_pull_log(pull_log, label=label)
    labels = [label]
    tc_steps = {}
    if tc_delay: tc_steps["delay"] = load_tc_log(tc_delay)
    if tc_loss:  tc_steps["loss"]  = load_tc_log(tc_loss)
    print(f"Single run '{label}': utility rows={len(df_util)}, monitor rows={len(df_mon)}")

df_util.head(3)

## 3  Quick stats table

In [ ]:
stats = (
    df_util
    .groupby(["label", "path"])[["bw_mbps", "owd_ms", "U", "loss", "G", "D", "L"]]
    .agg(["mean", "std", "max"])
    .round(3)
)
stats

## 4  Figure 1 — Bandwidth time series

In [ ]:
COLORS = {"baseline": "#2196F3", "delay": "#FF5722", "loss": "#4CAF50"}
LSTYLE = {"baseline": "-", "delay": "--", "loss": "-."}

fig, ax = plt.subplots(figsize=(10, 4))
for label in labels:
    sub = df_util[df_util["label"] == label]
    for path in ACTIVE_PATHS:
        s = sub[sub["path"] == path].groupby("t")["bw_mbps"].mean()
        if s.empty: continue
        ax.plot(s.index, smooth(s), color=COLORS.get(label, "k"),
                linestyle=LSTYLE.get(label, "-"), lw=1.3,
                label=f"{label} path={path}")

# tc step markers
for tc_label, col, color in [("delay", "delay_ms", "#FF5722"), ("loss", "loss_pct", "#4CAF50")]:
    if tc_label in tc_steps:
        for _, row in tc_steps[tc_label].iterrows():
            ax.axvline(row["at_sec"], lw=0.7, ls=":", color=color)

ax.set_xlabel("Time (s)"); ax.set_ylabel("Bandwidth (Mbps)")
ax.set_title("Path bandwidth over time")
ax.legend(fontsize=8, ncol=3); ax.grid(lw=0.4, alpha=0.5)
plt.tight_layout()

## 5  Figure 2 — One-Way Delay (OWD)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label in labels:
    sub = df_util[df_util["label"] == label]
    for path in ACTIVE_PATHS:
        s = sub[sub["path"] == path].groupby("t")["owd_ms"].mean()
        if s.empty: continue
        ax.plot(s.index, smooth(s), color=COLORS.get(label, "k"),
                linestyle=LSTYLE.get(label, "-"), lw=1.3,
                label=f"{label} path={path}")

if "delay" in tc_steps:
    for _, row in tc_steps["delay"].iterrows():
        ax.axvline(row["at_sec"], lw=0.7, ls=":", color="#FF5722")
        ax.text(row["at_sec"]+0.5, ax.get_ylim()[1]*0.9,
                f"{row['delay_ms']:.0f}ms", color="#FF5722", fontsize=7, rotation=90, va="top")

ax.set_xlabel("Time (s)"); ax.set_ylabel("OWD (ms)")
ax.set_title("One-way delay — delay-step runs show step increase")
ax.legend(fontsize=8, ncol=3); ax.grid(lw=0.4, alpha=0.5)
plt.tight_layout()

## 6  Figure 3 — Utility U

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label in labels:
    sub = df_util[df_util["label"] == label]
    for path in ACTIVE_PATHS:
        s = sub[sub["path"] == path].groupby("t")["U"].mean()
        if s.empty: continue
        ax.plot(s.index, smooth(s), color=COLORS.get(label, "k"),
                linestyle=LSTYLE.get(label, "-"), lw=1.3,
                label=f"{label} path={path}")

for tc_label, color in [("delay", "#FF5722"), ("loss", "#4CAF50")]:
    if tc_label in tc_steps:
        for _, row in tc_steps[tc_label].iterrows():
            ax.axvline(row["at_sec"], lw=0.7, ls=":", color=color)

ax.set_xlabel("Time (s)"); ax.set_ylabel("Utility U")
ax.set_title("4D-MAP utility score U over time")
ax.legend(fontsize=8, ncol=3); ax.grid(lw=0.4, alpha=0.5)
plt.tight_layout()

## 7  Figure 4 — Loss rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for label in labels:
    sub = df_util[df_util["label"] == label]
    for path in ACTIVE_PATHS:
        s = sub[sub["path"] == path].groupby("t")["loss"].mean()
        if s.empty: continue
        ax.plot(s.index, smooth(s), color=COLORS.get(label, "k"),
                linestyle=LSTYLE.get(label, "-"), lw=1.3,
                label=f"{label} path={path}")

if "loss" in tc_steps:
    for _, row in tc_steps["loss"].iterrows():
        ax.axvline(row["at_sec"], lw=0.7, ls=":", color="#4CAF50")
        ax.text(row["at_sec"]+0.5, ax.get_ylim()[1]*0.9,
                f"{row['loss_pct']:.1f}%", color="#4CAF50", fontsize=7, rotation=90, va="top")

ax.set_xlabel("Time (s)"); ax.set_ylabel("Loss rate")
ax.set_title("Packet loss rate — loss-step runs show step increase")
ax.legend(fontsize=8, ncol=3); ax.grid(lw=0.4, alpha=0.5)
plt.tight_layout()

## 8  Figure 5 — Utility component decomposition (G / D / L)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 10), sharex=True)
for ax, (comp, title) in zip(axes, [
    ("G", "G — throughput component"),
    ("D", "D — delay component"),
    ("L", "L — loss component"),
]):
    for label in labels:
        sub = df_util[df_util["label"] == label]
        for path in ACTIVE_PATHS:
            s = sub[sub["path"] == path].groupby("t")[comp].mean()
            if s.empty: continue
            ax.plot(s.index, smooth(s), color=COLORS.get(label, "k"),
                    linestyle=LSTYLE.get(label, "-"), lw=1.2,
                    label=f"{label} p{path}")
    for tc_label, color in [("delay", "#FF5722"), ("loss", "#4CAF50")]:
        if tc_label in tc_steps:
            for _, row in tc_steps[tc_label].iterrows():
                ax.axvline(row["at_sec"], lw=0.7, ls=":", color=color)
    ax.set_ylabel(title, fontsize=9); ax.grid(lw=0.4, alpha=0.5)
    ax.legend(fontsize=7, ncol=3)
axes[-1].set_xlabel("Time (s)")
fig.suptitle("Utility decomposition: how each component responds", fontsize=12)
plt.tight_layout(rect=[0,0,1,0.97])

## 9  Figure 6 — CWND & inflight (from [m]monitor)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for ax, (metric, ylabel) in zip(axes, [
    ("cwnd_room", "CWND room (bytes)"),
    ("inflight",  "Inflight (bytes)"),
]):
    for label in labels:
        sub = df_mon[df_mon["label"] == label]
        for path in ACTIVE_PATHS:
            s = sub[sub["path"] == path].groupby("t")[metric].mean()
            if s.empty: continue
            ax.plot(s.index, smooth(s), color=COLORS.get(label, "k"),
                    linestyle=LSTYLE.get(label, "-"), lw=1.2,
                    label=f"{label} p{path}")
    ax.set_ylabel(ylabel, fontsize=9); ax.grid(lw=0.4, alpha=0.5)
    ax.legend(fontsize=7, ncol=3)
axes[-1].set_xlabel("Time (s)")
fig.suptitle("QUIC congestion window — monitor view", fontsize=12)
plt.tight_layout(rect=[0,0,1,0.97])

## 10  Save all figures to PDF

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

out_pdf = REPO / "figures" / "phase2_analysis.pdf"
out_pdf.parent.mkdir(exist_ok=True)

# re-run all figures and collect in PDF
# Just open the PDF and paste each figure cell output in a loop
print(f"Tip: in JupyterLab, File > Export as > PDF to save all cells.")
print(f"Or call: python3 scripts/analyze/plot_phase2.py --single {SINGLE_RUN_DIR}")
print(f"Figures will be saved under: {REPO / 'figures'}")